<a href="https://colab.research.google.com/github/rohini-th/Ai/blob/main/final2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import nltk
from  nltk.tokenize import word_tokenize


from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Input,LSTM,Embedding,Dropout
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv(
    "/content/Reviews.csv",
    usecols=["Text"],
    nrows=10000,
    engine="python",
    on_bad_lines="skip"
)

In [3]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

df["Text"] = df["Text"].apply(clean_text)
df = df[df["Text"] != ""]

In [8]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(df["Text"])

print(tokenizer.word_index)

{'the': 1, 'i': 2, 'and': 3, 'a': 4, 'it': 5, 'to': 6, 'of': 7, 'is': 8, 'this': 9, 'br': 10, 'for': 11, 'in': 12, 'my': 13, 'that': 14, 'but': 15, 'you': 16, 'with': 17, 'not': 18, 'have': 19, 's': 20, 'was': 21, 'are': 22, 'they': 23, 't': 24, 'as': 25, 'on': 26, 'like': 27, 'so': 28, 'these': 29, 'coffee': 30, 'good': 31, 'them': 32, 'be': 33, 'can': 34, 'taste': 35, 'or': 36, 'just': 37, 'one': 38, 'if': 39, 'at': 40, 'great': 41, 'very': 42, 'product': 43, 'all': 44, 'flavor': 45, 'we': 46, 'from': 47, 'more': 48, 'has': 49, 'had': 50, 'when': 51, 'will': 52, 'me': 53, 'would': 54, 'love': 55, 'than': 56, 'no': 57, 'other': 58, 'out': 59, 'really': 60, 'some': 61, 'amazon': 62, 'food': 63, 'only': 64, 'tea': 65, 'too': 66, 'up': 67, 'about': 68, 'get': 69, 'much': 70, 'don': 71, 'cup': 72, 'there': 73, 'use': 74, 'an': 75, 'also': 76, 'best': 77, 'were': 78, 'because': 79, 'little': 80, 'tried': 81, 'your': 82, 'time': 83, 'what': 84, 've': 85, 'buy': 86, 'price': 87, 'make': 88, 

In [9]:
print(len(tokenizer.word_index)+1)

18207


In [10]:
wordindex = tokenizer.word_index
reverse_word_index = {index:word for word,index in wordindex.items()}
print(reverse_word_index)


{1: 'the', 2: 'i', 3: 'and', 4: 'a', 5: 'it', 6: 'to', 7: 'of', 8: 'is', 9: 'this', 10: 'br', 11: 'for', 12: 'in', 13: 'my', 14: 'that', 15: 'but', 16: 'you', 17: 'with', 18: 'not', 19: 'have', 20: 's', 21: 'was', 22: 'are', 23: 'they', 24: 't', 25: 'as', 26: 'on', 27: 'like', 28: 'so', 29: 'these', 30: 'coffee', 31: 'good', 32: 'them', 33: 'be', 34: 'can', 35: 'taste', 36: 'or', 37: 'just', 38: 'one', 39: 'if', 40: 'at', 41: 'great', 42: 'very', 43: 'product', 44: 'all', 45: 'flavor', 46: 'we', 47: 'from', 48: 'more', 49: 'has', 50: 'had', 51: 'when', 52: 'will', 53: 'me', 54: 'would', 55: 'love', 56: 'than', 57: 'no', 58: 'other', 59: 'out', 60: 'really', 61: 'some', 62: 'amazon', 63: 'food', 64: 'only', 65: 'tea', 66: 'too', 67: 'up', 68: 'about', 69: 'get', 70: 'much', 71: 'don', 72: 'cup', 73: 'there', 74: 'use', 75: 'an', 76: 'also', 77: 'best', 78: 'were', 79: 'because', 80: 'little', 81: 'tried', 82: 'your', 83: 'time', 84: 'what', 85: 've', 86: 'buy', 87: 'price', 88: 'make', 

In [12]:
df["Text"][:10]

,Text
0,i have bought several of the vitality canned d...
1,product arrived labeled as jumbo salted peanut...
2,this is a confection that has been around a fe...
3,if you are looking for the secret ingredient i...
4,great taffy at a great price there was a wide ...
5,i got a wild hair for taffy and ordered this f...
6,this saltwater taffy had great flavors and was...
7,this taffy is so good it is very soft and chew...
8,right now i m mostly just sprouting this so my...
9,this is a very healthy dog food good for their...


In [15]:
token_list = tokenizer.texts_to_sequences(df["Text"])[0]
input_sequences = []
# create n gram sequences: we can create a 6 gram sequence: n=6
for i in range(5,len(token_list)):
  n_gram_sequence = token_list[i-5:i+1] #5-5=0 [0:6], 2nd loop [1:7],3rd loop: [2:8]
  input_sequences.append(n_gram_sequence)

print(input_sequences[:5])


[[2, 19, 126, 314, 7, 1], [19, 126, 314, 7, 1, 6857], [126, 314, 7, 1, 6857, 582], [314, 7, 1, 6857, 582, 145], [7, 1, 6857, 582, 145, 63]]


In [17]:
input_sequences = np.array(input_sequences)

# Split into input (X) and output (y)
X = input_sequences[:, :-1]
y = input_sequences[:, -1].astype("int32")

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (43, 5)
y shape: (43,)


In [20]:
model = Sequential()

# Correct input shape = sequence length - 1
model.add(Input(shape=(max_sequences - 1,))) # 5

# Embedding layer
model.add(Embedding(input_dim=len(tokenizer.word_index)+1, output_dim=128))

# First LSTM layer
model.add(LSTM(150, return_sequences=True, dropout=0.2))

# Second LSTM layer
model.add(LSTM(100, dropout=0.2))

# Hidden Dense layer
model.add(Dense(100, activation='relu'))

# Output layer
model.add(Dense(len(tokenizer.word_index)+1, activation='softmax')) #units=6032

model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 128)         │     2,330,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 18207)          │     1,838,907 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,447,303 (16.97 MB)

 Trainable params: 4,447,303 (16.97 MB)

 Non-trainable params: 0 (0.00 B)

In [21]:
early_stop = EarlyStopping(monitor='loss',patience=3,restore_best_weights=True)

history = model.fit(X,y,epochs=30,batch_size=32,verbose=1,callbacks=[early_stop])

Epoch 1/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 125ms/step - accuracy: 0.0000e+00 - loss: 9.8094
Epoch 2/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.0930 - loss: 9.8052
Epoch 3/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - accuracy: 0.0698 - loss: 9.8004
Epoch 4/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.0698 - loss: 9.7946
Epoch 5/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 83ms/step - accuracy: 0.0698 - loss: 9.7860
Epoch 6/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.0465 - loss: 9.7730
Epoch 7/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 91ms/step - accuracy: 0.0930 - loss: 9.7504
Epoch 8/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 99ms/step - accuracy: 0.0698 - loss: 9.7077
Epoch 9/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 82ms/step - accuracy: 0.0465 - loss: 9.6279
Epoch 10/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 86ms/step - accuracy: 0.0465 - loss: 9.4706
Epoch 11/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.0465 - loss: 9.1691
Epoch 12/30
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 85ms/step - accuracy: 0.0465 - loss: 8.5

In [22]:
model.save("TextGenerationModel.keras")

In [23]:
def sample_with_temperature(preds, temperature=0.8, top_k=5):
    preds = np.asarray(preds).astype("float64")
    # [0.87,0.09,0.56,0.44,0.37,.............,0.89,0.32,...]

    # Select top k probabilities
    top_indices = np.argsort(preds)[-top_k:]
    # argsort : [0.09,0.32,0.37,0.44,0.56,0.87,0.89,........]
    # index of top k proabilities : [index]
    top_probs = preds[top_indices]
    #top k probs

    # Apply temperature scaling
    top_probs = np.log(top_probs + 1e-10) / temperature
    exp_probs = np.exp(top_probs)
    top_probs = exp_probs / np.sum(exp_probs)

    return np.random.choice(top_indices, p=top_probs)

In [30]:
def generate_text(seed_text, next_words=20):
    output_text = seed_text
    generated_words = []

    for _ in range(next_words):

        token_list = tokenizer.texts_to_sequences([output_text])[0]

        token_list = pad_sequences(
            [token_list],
            maxlen=max_sequences - 1,
            padding='pre'
        )

        predicted_probs = model.predict(token_list, verbose=0)[0]

        attempts = 0
        while attempts < 3:
            predicted_index = sample_with_temperature(
                predicted_probs,
                temperature=0.8,
                top_k=10
            )

            next_word = reverse_word_index.get(predicted_index, "")

            if next_word != "" and next_word not in generated_words[-3:]:
                break

            attempts += 1

        generated_words.append(next_word)
        output_text += " " + next_word

    return output_text

In [31]:
print(generate_text("this food", 30))



print(generate_text("\n i love", 30))
print(generate_text("\n the taste", 30))
print(generate_text("\n Great food", 30))

this food and than a appreciates the than better better better and finicky than all and than better the better a a and quality all than a product and better finicky quality

 i love better a the than product a finicky better appreciates the all finicky appreciates better the and finicky quality a the and all appreciates than and than appreciates all better a

 the taste finicky and appreciates than a product better than finicky all a and quality and a all the better all finicky the quality than better better and the better than and

 Great food than better the appreciates and a all better the product than better than and and a appreciates quality and than a appreciates better product and than and finicky better quality


In [26]:
print(len(history.history['loss']))

21


In [27]:
print("Vocabulary Size:", len(tokenizer.word_index) + 1)

print("X shape:", X.shape)
print("y shape:", y.shape)

model.summary()

Vocabulary Size: 18207
X shape: (43, 5)
y shape: (43,)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 5, 128)         │     2,330,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 5, 150)         │       167,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 100)            │       100,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 100)            │        10,100 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 18207)          │     1,838,907 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,341,911 (50.90 MB)

 Trainable params: 4,447,303 (16.97 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,894,608 (33.93 MB)

In [28]:
print(df.shape)

(10000, 1)


In [29]:
print(len(input_sequences))

43
